<a href="https://colab.research.google.com/github/Surajsurya95096/My-Bot-Deployer/blob/main/deploy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 🚀 **Universal Heroku Deployer (.env Support)** { display-mode: "form" }

# @markdown ### ⚙️ Heroku Credentials
Heroku_Email = ""  # @param {type:"string"}
Heroku_API_Key = ""  # @param {type:"string"}
Heroku_App_Name = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### 🔒 GitHub Repository
Git_Repo_URL = ""  # @param {type:"string"}
Git_Branch = "master"  # @param {type:"string"}
GitHub_Personal_Access_Token = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### 📝 Environment Variables (.env Mode)
# @markdown Yahan apna poora .env content paste karein ya neeche Upload_ENV_File ko tick karein
ENV_Text = ""  # @param {type:"string"}
Upload_ENV_File = False  # @param {type:"boolean"}

# @markdown ---
# @markdown ### ⚙️ Dyno & Settings
Dyno_Type = "web"  # @param ["worker", "web"]
Auto_Scale = True  # @param {type:"boolean"}
Stream_Logs = True  # @param {type:"boolean"}

import os
import subprocess
import sys
import time
from google.colab import files

GREEN = "\033[92m"
RED = "\033[91m"
YELLOW = "\033[93m"
RESET = "\033[0m"

def run_cmd(cmd, check=True):
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and res.returncode != 0:
        print(f"{RED}Error: {res.stderr.strip()}{RESET}")
    return res.returncode, res.stdout, res.stderr

if not Heroku_Email or not Heroku_API_Key or not Heroku_App_Name or not Git_Repo_URL:
    print(f"{RED}[!] Saare required Heroku & Repo fields bharein!{RESET}")
    sys.exit(1)

# Handle .env data
env_lines = []
if Upload_ENV_File:
    print(f"{YELLOW}[*] Apni .env file select karke upload karein...{RESET}")
    uploaded = files.upload()
    for fn in uploaded.keys():
        env_lines = uploaded[fn].decode("utf-8").splitlines()
elif ENV_Text.strip():
    env_lines = ENV_Text.strip().splitlines()

formatted_repo_url = Git_Repo_URL.strip()
if GitHub_Personal_Access_Token.strip():
    clean_url = formatted_repo_url.replace("https://", "").replace("http://", "").split("@")[-1]
    formatted_repo_url = f"https://{GitHub_Personal_Access_Token.strip()}@{clean_url}"

print(f"{YELLOW}[1/5] Heroku CLI verify ho rahi hai...{RESET}")
subprocess.run("curl -s https://cli-assets.heroku.com/install.sh | sh > /dev/null 2>&1", shell=True)

print(f"{YELLOW}[2/5] Heroku authentication configure ho raha hai...{RESET}")
netrc_data = f"machine api.heroku.com\n  login {Heroku_Email.strip()}\n  password {Heroku_API_Key.strip()}\nmachine git.heroku.com\n  login {Heroku_Email.strip()}\n  password {Heroku_API_Key.strip()}\n"
with open(os.path.expanduser("~/.netrc"), "w") as f:
    f.write(netrc_data)
os.chmod(os.path.expanduser("~/.netrc"), 0o600)

app_name = Heroku_App_Name.strip().lower()
print(f"{YELLOW}[3/5] Heroku App setup ho rahi hai...{RESET}")
create_code, out, err = run_cmd(f"heroku create {app_name}", check=False)
if create_code != 0:
    run_cmd(f"heroku git:remote -a {app_name}")

# Push Config Vars to Heroku
if env_lines:
    print(f"{YELLOW}[*] .env Variables Heroku par set ho rahe hain...{RESET}")
    config_args = []
    for line in env_lines:
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, val = line.split("=", 1)
            val = val.strip().strip("'").strip('"')
            config_args.append(f'{key.strip()}="{val}"')
    if config_args:
        run_cmd(f"heroku config:set {' '.join(config_args)} -a {app_name}")
        print(f"{GREEN}✅ Saare .env variables successfully set ho gaye!{RESET}")

print(f"{YELLOW}[4/5] Repository clone & Heroku push shuru...{RESET}")
repo_dir = "/content/bot_deploy"
if os.path.exists(repo_dir):
    subprocess.run(f"rm -rf {repo_dir}", shell=True)

code, _, err = run_cmd(f"git clone -b {Git_Branch.strip()} {formatted_repo_url} {repo_dir}")
if code != 0:
    print(f"{RED}[❌] Git Clone fail ho gaya! Branch name ya token check karein.{RESET}")
    sys.exit(1)

os.chdir(repo_dir)
run_cmd('git config --global user.email "deployer@colab.local"')
run_cmd('git config --global user.name "Colab Deployer"')

print(f"{GREEN}[5/5] Heroku par code deploy ho raha hai...{RESET}\n")
deploy = subprocess.Popen(f"git push https://git.heroku.com/{app_name}.git HEAD:main --force", shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in deploy.stdout:
    print(line, end="")
deploy.wait()

if deploy.returncode == 0:
    print(f"\n{GREEN}✅ App Deployed Successfully!{RESET}")
    if Auto_Scale:
        run_cmd(f"heroku ps:scale {Dyno_Type}=1 -a {app_name}")
        print(f"{GREEN}Dyno Start ho gaya!{RESET}")
    if Stream_Logs:
        time.sleep(2)
        log_proc = subprocess.Popen(f"heroku logs --tail -a {app_name}", shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        try:
            for log_line in log_proc.stdout:
                print(log_line, end="")
        except KeyboardInterrupt:
            pass
else:
    print(f"\n{RED}❌ Build fail hui. Logs check karein.{RESET}")